# 01 — 사용자 페르소나 및 IAM 설정

이 Notebook에서는 AWS Agent Registry와 상호 작용하는 데 필요한 IAM 페르소나 역할을 프로비저닝하는 방법을 안내합니다. 이 역할을 통해 레지스트리 생성, 레코드 게시, 승인, 검색과 같은 핵심 레지스트리 작업을 수행할 수 있습니다.

## 학습 내용

- AWS Agent Registry의 개념
- 페르소나 역할(관리자, 게시자, 소비자)과 직무 분리가 중요한 이유
- 각 페르소나에 범위가 지정된 권한을 갖는 IAM 역할을 생성하는 방법
- 각 역할을 수임하는 방법
- 다음 Getting Started Notebook(02*)으로 진행하기 전에 설정을 검증하는 방법



---

### 사용 사례: 엔터프라이즈 결제 처리

#### 개요

AnyCompany는 매일 수천 건의 거래를 처리하는 전자 상거래 플랫폼과 함께 대출 신청, 신용 조회, 할부 상품을 처리하는 대출 부문을 운영합니다. 두 영역의 모든 AI 에이전트마다 별도의 통합을 구축하는 대신, 결제 및 대출 처리 기능을 게시하고 모든 에이전트가 런타임에 검색할 수 있는 중앙 레지스트리를 구축하려고 합니다.

#### 비즈니스 배경

| | |
|---|---|
| **과제** | 고객 서비스 에이전트에 실시간 결제 처리, 환불 처리, 거래 상태 조회 기능이 필요함 |
| **목표** | 모든 AI 에이전트가 검색하고 사용할 수 있도록 결제 처리 기능을 중앙 집중화 |
| **이점** | 통합 복잡성을 줄이고, 일관된 결제 처리를 보장하며, 에이전트, 스킬, MCP 및 사용자 지정 도구를 신속하게 배포 |

### 솔루션: AWS Agent Registry란?

[AWS Agent Registry](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry.html#registry-what-is)는 에이전트와 도구를 검색하고, 거버넌스를 적용하고, 관리하기 위한 통합 카탈로그입니다.

#### 핵심 기능

- **검색:** 에이전트와 도구를 검색하기 위한 하이브리드 검색(키워드 및 시맨틱 검색). 구조화된 속성에 대한 키워드 필터링을 지원합니다.
- **거버넌스:** 등록된 에이전트와 도구에 승인 워크플로를 통합하고 정책 검증을 적용합니다.
- **시맨틱 검색:** 소비자는 필요한 기능을 설명하여 적합한 도구를 찾을 수 있습니다.
- **보안:** Registry 및 Registry 레코드 조회/업데이트에 대한 보안 액세스를 제공합니다. IAM 및 OAuth를 사용한 계정 간 Registry 액세스를 지원합니다.
- **관찰성:** 사용 통계, 성능 지표, 오류율 및 규정 준수를 위한 감사 추적을 제공합니다(⚠️ 제공 예정).

## 엔드 투 엔드 워크플로
아래 다이어그램은 엔드 투 엔드 워크플로를 보여 줍니다.

1. **관리자가 레지스트리를 생성합니다.**
2. **게시자가 레지스트리 레코드를 생성합니다.** 레코드에는 A2A 에이전트, MCP 도구, Agent Skills 또는 사용자 지정 도구의 메타데이터가 포함됩니다.
3. **관리자가 제출된 레코드를 승인하거나 거부합니다.**
4. **소비자가 AWS Console, Kiro 또는 SDK를 통해 승인된 레코드를 검색하고 찾습니다.**

![AgentCore Registry 개요](images/registry-architecture.png)


## 레지스트리 페르소나 역할

이 워크숍에서는 직무 분리를 보여 주기 위해 세 가지 IAM 역할을 사용합니다.

| 페르소나 | 역할 이름 | 용도 | Notebook |
|---------|-----------|---------|----------|
| **관리자** | `admin_persona` | 레지스트리 생성, 레코드 승인/거부, 워크로드 자격 증명 관리 | 02, 04 |
| **게시자** | `publisher_persona` | 레지스트리 레코드를 생성하고 승인을 요청 | 03 |
| **소비자** | `consumer_persona` | 승인된 레코드를 검색하고 탐색 | 05 |

각 역할에는 필요한 권한만 부여되며, 그 이상의 권한은 부여되지 않습니다.

## Notebook 시리즈

```
01 IAM 설정               페르소나 역할 및 권한 생성
 └─▶ 02 레지스트리 생성       관리자가 승인 워크플로를 사용하는 레지스트리 생성
      └─▶ 03 레코드 게시          게시자가 도구 레코드 제출
           └─▶ 04 승인 워크플로       관리자가 제출 항목 승인/거부
                └─▶ 05 시맨틱 검색       소비자가 승인된 도구 검색
```

위 순서대로 Notebook을 실행하세요. 이 **Notebook(01)** 을 먼저 완료해야 합니다.

## 현재 자격 증명이 사용되는 방식

이 Notebook은 현재 사용 중인 자격 증명(IAM 역할/사용자, SageMaker 역할)을 감지하고 다음 작업을 수행합니다.

1. 세 가지 IAM 역할(관리자, 게시자, 소비자)을 생성합니다.
2. 현재 자격 증명을 각 페르소나 역할의 신뢰 정책에 **신뢰할 수 있는 보안 주체**로 추가하여, `sts:AssumeRole`을 호출해 어떤 역할로든 전환할 수 있도록 합니다.
3. 각 페르소나 역할에 범위가 지정된 `bedrock-agentcore` 작업을 허용하는 **인라인 권한 정책**을 연결합니다.

이후 각 Notebook(02–05)은 `sts.assume_role()`을 통해 관련 페르소나 역할을 수임합니다. 따라서 모든 API 호출은 해당 페르소나에 필요한 권한으로만 실행되며, 이는 프로덕션에서 각기 다른 팀이 운영하는 방식을 재현합니다.

## 사전 요구 사항

- **boto3 >= 1.42.87**
- **IAM 권한** — 사용 중인 IAM 사용자 또는 역할에는 역할 생성, 정책 연결, 역할 수임 권한이 필요합니다. 필요한 최소 정책은 아래와 같습니다.

> **참고:** 현재 자격 증명에는 `bedrock-agentcore` 권한이 필요하지 않습니다. 이 권한은 이 Notebook에서 생성하는 페르소나 역할에 연결됩니다. 현재 자격 증명에는 IAM 관리 및 STS 권한만 필요합니다.

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "IAMRoleManagement",
            "Effect": "Allow",
            "Action": [
                "iam:CreateRole",
                "iam:GetRole",
                "iam:PutRolePolicy",
                "iam:UpdateAssumeRolePolicy",
                "iam:DeleteRole",
                "iam:DeleteRolePolicy",
                "iam:ListRolePolicies",
                "iam:ListAttachedRolePolicies",
                "iam:DetachRolePolicy",
                "iam:ListInstanceProfilesForRole",
                "iam:RemoveRoleFromInstanceProfile",
                "iam:PassRole"
            ],
            "Resource": "arn:aws:iam::*:role/*"
        },
        {
            "Sid": "STS",
            "Effect": "Allow",
            "Action": [
                "sts:AssumeRole",
                "sts:GetCallerIdentity"
            ],
            "Resource": "*"
        }
    ]
}
```


---
## 1. boto3 SDK 및 종속성 설치

필요한 Python 패키지를 설치합니다.

In [ ]:
!pip install boto3 python-dotenv --force-reinstall

## 2. 현재 자격 증명 자동 감지

이 코드는 **Amazon SageMaker Notebook**(역할이 자동으로 연결됨)과 AWS 자격 증명이 구성된 환경에서 모두 작동합니다. SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명이 설정되어 있는지 확인하세요(예: `aws configure`, 환경 변수 또는 자격 증명 파일).

현재 호출자 자격 증명을 감지하고 IAM 보안 주체 ARN을 추출합니다. 수임된 역할 또는 서비스 역할(예: SageMaker)로 실행되는 경우, ARN은 신뢰 정책에 필요한 기본 IAM 역할 ARN 형식으로 변환됩니다.

In [ ]:
import boto3
import time
import botocore.exceptions
import utils
import os

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
ACCOUNT_ID = ""
try:
    sts = boto3.client("sts", region_name=AWS_REGION)
    identity = sts.get_caller_identity()
    ACCOUNT_ID = identity["Account"]
    CALLER_ARN = identity["Arn"]
    print(f"Account ID : {ACCOUNT_ID}")
    print(f"Caller ARN : {CALLER_ARN}")
except (botocore.exceptions.NoCredentialsError, botocore.exceptions.ClientError) as e:
    print(f"ERROR: Could not retrieve caller identity — {e}")
    print("Make sure your AWS credentials are configured.")
    raise SystemExit(1)

호출자 ARN을 기본 IAM 보안 주체 ARN으로 변환합니다. 이 코드는 수임된 역할 및 서비스 역할 ARN 형식을 처리합니다.

In [ ]:
ROLE_ARN = utils.extract_role_arn(CALLER_ARN)
print(f"Principal ARN: {ROLE_ARN}")

역할 및 정책 관리를 위한 IAM 클라이언트를 생성합니다.

In [ ]:
iam_client = boto3.client("iam", region_name=AWS_REGION)
print(f"IAM client ready — region: {AWS_REGION}")

## 3. 페르소나 역할 생성

### 페르소나 권한 정의

각 페르소나는 정책 이름과 허용된 `bedrock-agentcore` 작업 목록으로 정의됩니다. **관리자**는 레지스트리를 완전히 제어할 수 있고, **게시자**는 레코드를 생성하고 제출할 수 있으며, **소비자**는 검색과 조회만 할 수 있습니다.

In [ ]:
PERSONA_DEFINITIONS = {
    "admin_persona": {
        "policy_name": "AdminPolicy",
        "actions": [
            "bedrock-agentcore:CreateRegistry",
            "bedrock-agentcore:ListRegistries",
            "bedrock-agentcore:GetRegistry",
            "bedrock-agentcore:UpdateRegistry",
            "bedrock-agentcore:DeleteRegistry",
            "bedrock-agentcore:CreateRegistryRecord",
            "bedrock-agentcore:ListRegistryRecords",
            "bedrock-agentcore:GetRegistryRecord",
            "bedrock-agentcore:UpdateRegistryRecord",
            "bedrock-agentcore:DeleteRegistryRecord",
            "bedrock-agentcore:SubmitRegistryRecordForApproval",
            "bedrock-agentcore:UpdateRegistryRecordStatus",
            "bedrock-agentcore:*WorkloadIdentity",
        ],
    },
    "publisher_persona": {
        "policy_name": "PublisherPolicy",
        "actions": [
            "bedrock-agentcore:ListRegistries",
            "bedrock-agentcore:GetRegistry",
            "bedrock-agentcore:CreateRegistryRecord",
            "bedrock-agentcore:ListRegistryRecords",
            "bedrock-agentcore:GetRegistryRecord",
            "bedrock-agentcore:DeleteRegistryRecord",
            "bedrock-agentcore:UpdateRegistryRecord",
            "bedrock-agentcore:SubmitRegistryRecordForApproval",
        ],
    },
    "consumer_persona": {
        "policy_name": "ConsumerPolicy",
        "actions": [
            "bedrock-agentcore:ListRegistries",
            "bedrock-agentcore:GetRegistry",
            "bedrock-agentcore:GetRegistryRecord",
            "bedrock-agentcore:ListRegistryRecords",
            "bedrock-agentcore:SearchRegistryRecords",
        ],
    },
}

### 모든 페르소나 역할 생성 또는 업데이트

각 페르소나 정의를 순회하면서 역할을 생성하고(이미 있는 경우 업데이트) 권한 정책을 연결합니다. 역할 생성 사이에 잠시 대기하여 IAM 스로틀링을 방지합니다.

In [ ]:
trust_policy = utils.build_trust_policy(ROLE_ARN)
persona_role_arns = {}

for role_name, config in PERSONA_DEFINITIONS.items():
    print(f"\n{'=' * 60}")
    print(f"Setting up: {role_name}")
    print(f"{'=' * 60}")
    role_arn = utils.create_or_update_persona_role(
        iam_client,
        role_name,
        config["policy_name"],
        config["actions"],
        trust_policy,
        ACCOUNT_ID,
    )
    persona_role_arns[role_name] = role_arn
    time.sleep(1)  # 역할 생성 사이에 잠시 대기

print("\n✅ All persona roles ready:")
for name, arn in persona_role_arns.items():
    print(f"  {name}: {arn}")

---
## 4. 검증 — 각 페르소나 역할 수임 테스트

IAM 변경 사항이 전파되도록 10초 동안 기다린 다음 각 페르소나 역할 수임을 테스트합니다. 세 역할 모두 성공해야 합니다.

In [ ]:
print("Waiting 10 seconds for IAM propagation...")
time.sleep(10)
print("Done.")

### 역할 수임 테스트

`sts.assume_role()`을 사용하여 각 페르소나 역할의 수임을 시도합니다. 수임에 성공하면 신뢰 정책과 AssumeRole 권한이 올바르게 구성된 것입니다.

In [ ]:
results = {}

for role_name, role_arn in persona_role_arns.items():
    print(f"\nAssuming {role_name} ({role_arn})...")
    try:
        resp = utils.assume_role_only(AWS_REGION, role_arn, session_name=f"{role_name}-verify")
        utils.pp(resp)
        assumed_arn = resp["AssumedRoleUser"]["Arn"]
        expiration = resp["Credentials"]["Expiration"]
        print(f"  ✅ Success — assumed: {assumed_arn}")
        print(f"     Expires: {expiration}")
        results[role_name] = "PASS"
    except botocore.exceptions.ClientError as e:
        print(f"  ❌ Failed — {e}")
        print("     Try waiting a few more seconds for IAM propagation and re-run this cell.")
        results[role_name] = "FAIL"

print(f"\n{'=' * 60}")
print("Verification Summary")
print(f"{'=' * 60}")
for name, status in results.items():
    icon = "✅" if status == "PASS" else "❌"
    print(f"  {icon} {name}: {status}")

if all(s == "PASS" for s in results.values()):
    print("\n🎉 All roles verified! You are ready to proceed to notebook 02.")
else:
    print("\n⚠️  Some roles failed. Check the errors above and re-run after a brief wait.")

---
## 5. 수동 IAM 설정(대안)

프로그래밍 방식으로 역할을 생성할 수 없는 경우(예: 제한된 환경) AWS IAM Console에서 다음 단계를 수행하세요.

### 1단계: 각 페르소나 역할 생성

세 역할(`admin_persona`, `publisher_persona`, `consumer_persona`) 각각에 대해 다음을 수행합니다.

1. **IAM → Roles → Create role**로 이동합니다.
2. **Custom trust policy**를 선택합니다.
3. 아래의 신뢰 정책 JSON을 붙여 넣습니다(`<YOUR_IDENTITY_ARN>`을 IAM 사용자 또는 역할 ARN으로 바꿈).
4. **Next**를 클릭한 다음 **Create policy**를 클릭하여 권한 정책을 추가합니다.
5. 표시된 역할 이름을 정확히 사용합니다.

### 신뢰 정책(세 역할에 동일하게 적용)

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "Service": "bedrock-agentcore.amazonaws.com"
      },
      "Action": "sts:AssumeRole"
    },
    {
      "Effect": "Allow",
      "Principal": {
        "AWS": "<YOUR_IDENTITY_ARN>"
      },
      "Action": "sts:AssumeRole"
    }
  ]
}
```

### 2단계: 권한 정책 연결

#### AdminPolicy(`admin_persona`용)

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "bedrock-agentcore:CreateRegistry",
        "bedrock-agentcore:ListRegistries",
        "bedrock-agentcore:GetRegistry",
        "bedrock-agentcore:UpdateRegistry",
        "bedrock-agentcore:DeleteRegistry",
        "bedrock-agentcore:CreateRegistryRecord",
        "bedrock-agentcore:ListRegistryRecords",
        "bedrock-agentcore:GetRegistryRecord",
        "bedrock-agentcore:UpdateRegistryRecord",
        "bedrock-agentcore:DeleteRegistryRecord",
        "bedrock-agentcore:SubmitRegistryRecordForApproval",
        "bedrock-agentcore:UpdateRegistryRecordStatus",
        "bedrock-agentcore:*WorkloadIdentity"
      ],
      "Resource": "*"
    }
  ]
}
```

#### PublisherPolicy(`publisher_persona`용)

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "bedrock-agentcore:ListRegistries",
        "bedrock-agentcore:GetRegistry",
        "bedrock-agentcore:CreateRegistryRecord",
        "bedrock-agentcore:ListRegistryRecords",
        "bedrock-agentcore:GetRegistryRecord",
        "bedrock-agentcore:DeleteRegistryRecord",
        "bedrock-agentcore:UpdateRegistryRecord",
        "bedrock-agentcore:SubmitRegistryRecordForApproval"
      ],
      "Resource": "*"
    }
  ]
}
```

#### ConsumerPolicy(`consumer_persona`용)

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "bedrock-agentcore:ListRegistries",
        "bedrock-agentcore:GetRegistry",
        "bedrock-agentcore:GetRegistryRecord",
        "bedrock-agentcore:ListRegistryRecords",
        "bedrock-agentcore:SearchRegistryRecords"
      ],
      "Resource": "*"
    }
  ]
}
```

---
## 6. 정리

⚠️ **경고: 모든 Notebook(02–05)을 완료한 후에만 이 섹션을 실행하세요.** 이 역할을 삭제하면 다른 Notebook이 작동하지 않습니다.

In [ ]:
# cleanup_results = {}

# for role_name, config in PERSONA_DEFINITIONS.items():
#     print(f"\nCleaning up: {role_name}")
#     try:
#         # 1. 모든 인라인 정책 삭제(여기서 생성한 정책만이 아님)
#         try:
#             inline_policies = iam_client.list_role_policies(RoleName=role_name)
#             for policy_name in inline_policies.get("PolicyNames", []):
#                 iam_client.delete_role_policy(
#                     RoleName=role_name,
#                     PolicyName=policy_name,
#                 )
#                 print(f"  Deleted inline policy: {policy_name}")
#         except iam_client.exceptions.NoSuchEntityException:
#             print(f"  No inline policies found")

#         # 2. 관리형 정책 분리
#         try:
#             attached = iam_client.list_attached_role_policies(RoleName=role_name)
#             for policy in attached.get("AttachedPolicies", []):
#                 iam_client.detach_role_policy(
#                     RoleName=role_name,
#                     PolicyArn=policy["PolicyArn"],
#                 )
#                 print(f"  Detached managed policy: {policy['PolicyArn']}")
#         except iam_client.exceptions.NoSuchEntityException:
#             pass

#         # 3. 인스턴스 프로파일에서 제거
#         try:
#             profiles = iam_client.list_instance_profiles_for_role(RoleName=role_name)
#             for profile in profiles.get("InstanceProfiles", []):
#                 iam_client.remove_role_from_instance_profile(
#                     InstanceProfileName=profile["InstanceProfileName"],
#                     RoleName=role_name,
#                 )
#                 print(f"  Removed from instance profile: {profile['InstanceProfileName']}")
#         except iam_client.exceptions.NoSuchEntityException:
#             pass

#         # 4. 역할 삭제
#         iam_client.delete_role(RoleName=role_name)
#         print(f"  ✅ Deleted role: {role_name}")
#         cleanup_results[role_name] = "DELETED"

#     except iam_client.exceptions.NoSuchEntityException:
#         print(f"  Role {role_name} not found — already absent")
#         cleanup_results[role_name] = "ABSENT"
#     except botocore.exceptions.ClientError as e:
#         print(f"  ❌ Error: {e}")
#         cleanup_results[role_name] = "ERROR"

### 정리 요약

성공적으로 삭제된 리소스와 이미 존재하지 않았던 리소스를 표시합니다.

In [ ]:
# print(f"\n{'='*60}")
# print("Cleanup Summary")
# print(f"{'='*60}")
# for name, status in cleanup_results.items():
#     icon = "✅" if status == "DELETED" else "⚪" if status == "ABSENT" else "❌"
#     print(f"  {icon} {name}: {status}")
# print(f"\nCleanup complete.")

---
## 다음 단계

세 가지 페르소나 역할을 생성하고 검증했으므로, **Notebook 02 — 레지스트리 생성**으로 이동하여 관리자 페르소나로 첫 번째 AWS Agent Registry를 생성하세요.

여기서 생성한 역할은 나머지 모든 Notebook에서 사용됩니다.

- **Notebook 02** — [레지스트리 생성](02-creating-registry-workflow.ipynb): 관리자가 레지스트리를 생성합니다.
- **Notebook 03** — [레코드 게시](03-publishing-records-workflow.ipynb): 게시자로 레코드를 게시합니다.
- **Notebook 04** — [관리자 승인](04-admin-approval-workflow.ipynb): 관리자 승인 워크플로
- **Notebook 05** — [시맨틱 검색](05-search-registry-workflow.ipynb): 소비자로서 NLQ를 사용해 승인된 레코드를 검색합니다.